In [ ]:
!pip install -q langchain langchain-openai

In [ ]:
import os

from IPython.display import HTML, Image
from google.colab import userdata
from langchain.messages import HumanMessage, SystemMessage
from langchain_core.messages import BaseMessage
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import Runnable, RunnableConfig
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.base import BaseCheckpointSaver
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph, add_messages
from langgraph.graph.state import CompiledStateGraph
from langgraph.types import Command, Interrupt, Send, interrupt
from pathlib import Path
from pydantic import BaseModel, SecretStr
from typing import Annotated, Dict, List, Literal, TypedDict

openai_api_key = SecretStr(userdata.get('OPENAI_API_KEY'))
openrouter_api_key = SecretStr(userdata.get('OPENROUTER_API_KEY'))

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = userdata.get("LANGSMITH_API_KEY")
os.environ["LANGSMITH_PROJECT"] = "Marketing Campaign Squad"

def print_interrupts(interrupts: List[Interrupt]):
    for el in interrupts:
        display(HTML(f'<div style="border: 1px dashed red; margin: 5px; padding: 10px; white-space: pre-wrap;">{el.value}</div>'))

def display_graph(runnable: Runnable, output_png: Path) -> None:
    with output_png.open(mode="wb") as file:
        file.write(runnable.get_graph(xray=True).draw_mermaid_png())

    display(Image(output_png, format="png"))

def explore_checkpoints(checkpoint_saver: BaseCheckpointSaver, config: RunnableConfig):
    checkpoints = list(checkpoint_saver.list(config))
    print(f"There are {len(checkpoints)} checkpoints in total:")
    for checkpoint in reversed(checkpoints):
        print(checkpoint)

def explore_state_history(compiled_state_graph: CompiledStateGraph, config: RunnableConfig):
    state_history = list(compiled_state_graph.get_state_history(config))

    for snapshot in reversed(state_history):
        print(f"Step: {snapshot.metadata['step']}")
        print("Current state:")
        print(snapshot.values)
        print(f"Next: {snapshot.next}")
        print()

In [ ]:
class ReviewScore(BaseModel):
    criterion: str
    score: Literal["pass", "fail"]

# IDEA: Add a new field `is_automated`.
class EditorReview(BaseModel):
    verdict: Literal["approve", "reject"]
    scores: List[ReviewScore]
    revision_notes: str

In [ ]:
trend_analyst_model = ChatOpenAI(model="gpt-5-nano", api_key=openai_api_key, reasoning_effort="low")
trend_analyst_prompt = """You are the Trend Analyst on a social media campaign team.

ROLE
Your job is to produce a concise research brief that can be used to draft posts.
You do NOT write marketing copy yourself — your output is intelligence, not creative.

PLATFORM CONTEXT
You are researching specifically for {platform}.
All research dimensions below must be filtered through this platform lens.
A pain point or a hashtag strategy that works on one platform may be wrong for other. Tailor everything to {platform}.

INPUT
You will receive a product brief describing what is being launched and the intended audience.

TASK
Analyze the brief and produce research covering:
1. Audience profile — who they are beyond the surface description (demographics, psychographics, where they spend time online, what they distrust)
2. Pain points — 3-5 concrete problems this audience has that the product plausibly addresses
3. Cultural trends — 2-4 current movements, conversations, or aesthetics in this niche the campaign could ride
4. Hashtags — a tiered list: 2-3 broad/high-volume, 3-5 mid-tier niche, 2-3 community-specific
5. Tone signals — what voice resonates with this audience (e.g., "dry humor over earnestness", "data over hype")
6. Format guidance — what post format performs best on {platform} for this kind of campaign (e.g., carousel vs. single image, thread vs. standalone, short-form video vs. static). Include length expectations.
7. Things to avoid — clichés, overused phrases, or angles that have been done to death in this space

CONSTRAINTS
- Your research MUST be based on plausible facts about the market. Do not invent statistics or cite specific sources you cannot verify.
- Stay neutral and analytical. No marketing voice, no hype words, no exclamation marks.
"""

copywriter_model = ChatOpenAI(model="google/gemma-4-26b-a4b-it", api_key=openrouter_api_key, reasoning_effort="low", base_url="https://openrouter.ai/api/v1")
copywriter_prompt = """You are the Copywriter on a social media campaign team.

ROLE
You turn the product brief and the Trend Analyst's research into engaging, ready-to-post social media copy.

PLATFORM CONTEXT
You are writing exclusively for {platform}.
Every choice — length, tone, structure, hashtag usage, call-to-action style — must fit this platform's native conventions:
A draft that would work on a different platform is a failed draft, even if the writing is good.
The Trend Analyst has already filtered their research through the lens of the platform; trust their format guidance and tone signals.

INPUT
You will receive:
- product_brief: the original user request
- research: the Trend Analyst's structured findings
- revision_notes (optional): specific feedback from the QA Editor on a previous draft. If present, this is a REVISION pass — your job is to address each note explicitly.

TASK
Produce a social media post draft that:
- Opens with a hook in the first line (no warm-up, no "In today's world...")
- Speaks directly to one of the pain points from the research
- Matches the tone signals identified by the Analyst
- Uses hashtags from the research — 3-6, mixed tiers, placed at the end
- Includes one clear call-to-action
- Avoids the "things to avoid" list from the research

CONSTRAINTS
- Write the post itself in the voice of the brand, not in your own analytical voice.
- No clichés: "game-changer", "revolutionary", "unleash", "elevate your [X]", "in today's fast-paced world", "more than just a [product]". If you catch yourself reaching for one, rewrite the line.
- On revision passes, address every item in revision_notes. Do not silently skip feedback.
- One post, not three. Quality over variants.
"""

editor_model = ChatOpenAI(model="gpt-5.4-mini", api_key=openai_api_key, reasoning_effort="low").with_structured_output(EditorReview)
editor_prompt = """You are the Editor on a social media campaign team.

ROLE
You are the quality gate. You critique the Copywriter's draft against a fixed rubric and decide whether it ships or goes back for revision. You are skeptical by default — "fine" is not "approved".

PLATFORM CONTEXT
You are evaluating exclusively against the norms of {platform}.
Your job is not just "is this good writing" — it is "is this the right post for {platform}".

INPUT
You will receive:
- Product brief: the original request
- Research: the Trend Analyst's findings (use this as ground truth for tone and audience)
- Draft: the Copywriter's current draft
- Count of revisions: how many times this draft has cycled (use this to calibrate, not to lower standards)

EVALUATION RUBRIC
Score the draft on each dimension. Each must pass for overall approval.

1. Tone match — Does the voice align with the tone signals in the research?
2. Audience fit — Does it speak to the identified audience and at least one stated pain point?
3. Cliché-free — Is it free of generic marketing language? Flag specific phrases.
4. Hook strength — Does the first line earn the second line?
5. CTA clarity — Is there one unambiguous action the reader can take?
6. Format hygiene — Hashtag count and placement, length appropriate to platform, no broken punctuation.

DECISION
- APPROVE only if every dimension passes. Approval means this is publishable as-is.
- REJECT if any dimension fails. Rejection must come with specific, actionable feedback — not "make it better" but "the hook 'Stay hydrated, stay winning' is a cliché; rewrite around the post-workout recovery angle from the research".

OUTPUT FORMAT
Return JSON with:
- verdict: "approve" or "reject"
- scores: object with each rubric dimension as a key, value is "pass" or "fail"
- revision_notes: array of specific change requests (empty if approved). Each note should reference the exact phrase or issue and suggest a direction, not a rewrite.

CONSTRAINTS
- You do not rewrite the copy yourself. Your job is to diagnose, not to draft.
- Be specific. "Tone is off" is not feedback; "The phrase 'unleash your potential' clashes with the dry, data-driven tone the Analyst identified" is feedback.
- Do not lower the bar on later revision cycles. If the draft still fails on cycle 3, it still fails. Loop termination is the orchestrator's job, not yours.
"""

In [ ]:
class ContentCreationState(TypedDict):
    product_brief: str
    target_platform: str
    trend_analysis: str
    draft_messages: Annotated[List[BaseMessage], add_messages]
    draft: str
    review: EditorReview
    revision_cycles: int

In [ ]:
def analyze_trends(state: ContentCreationState):
    product_brief = state.get("product_brief")
    if not product_brief:
        raise RuntimeError("Cannot analyze trends: Missing product brief")

    platform = state.get("target_platform")
    if not platform:
        raise RuntimeError("Cannot analyze trends: Missing target platform")

    response = trend_analyst_model.invoke(
        input=[
            SystemMessage(PromptTemplate.from_template(trend_analyst_prompt).format(platform=platform)),
            HumanMessage(f"Product brief:\n{product_brief}")
        ]
    )

    return { "trend_analysis": response.content }

def copywrite(state: ContentCreationState):
    product_brief = state.get("product_brief")
    if not product_brief:
        raise RuntimeError("Cannot copywrite: Missing product brief")

    platform = state.get("target_platform")
    if not platform:
        raise RuntimeError("Cannot copywrite: Missing target platform")

    trend_analysis = state.get("trend_analysis")
    if not trend_analysis:
        raise RuntimeError("Cannot copywrite: Missing trend analysis")

    previous_messages = state.get("draft_messages", [])
    if previous_messages:
        review = state.get("review")
        if not review:
            raise RuntimeError("Cannot rework draft: Missing review")

        new_message = HumanMessage(f"Apply necessary corrections according to the review:\n{review.revision_notes}\n\nDo not comment the review.\nDo not summarize your changes.\nReturn only the reworked draft.")
    else:
        new_message = HumanMessage(f"Product brief:\n{product_brief}\n\nTrend analysis:\n{trend_analysis}")

    response = copywriter_model.invoke(
        input=[
            SystemMessage(PromptTemplate.from_template(copywriter_prompt).format(platform=platform)),
            *previous_messages,
            new_message
        ]
    )

    return { "draft": response.content, "draft_messages": [new_message, response] }

def review(state: ContentCreationState):
    product_brief = state.get("product_brief")
    if not product_brief:
        raise RuntimeError("Cannot review: Missing product brief")

    platform = state.get("target_platform")
    if not platform:
        raise RuntimeError("Cannot review: Missing target platform")

    trend_analysis = state.get("trend_analysis")
    if not trend_analysis:
        raise RuntimeError("Cannot review: Missing trend analysis")

    draft = state.get("draft")
    if not draft:
        raise RuntimeError("Cannot review: Missing draft")

    revision_cycles = state.get("revision_cycles", 0)
    if revision_cycles >= 2:
        human_response = interrupt(f"The automated review could not produce a concrete result within the configured threshold. Analyze the draft by yourself.\n\nPlatform: {platform}\n{draft}")
        review = EditorReview(verdict=human_response.get("decision"), scores=[], revision_notes=human_response.get("guidance", ""))
    else:
        review = editor_model.invoke(
            input=[
                SystemMessage(PromptTemplate.from_template(editor_prompt).format(platform=platform)),
                HumanMessage(f"Product brief:\n{product_brief}\n\nTrend analysis: {trend_analysis}\n\nDraft:\n{draft}\n\nRevision count: {revision_cycles}")
            ]
        )

    result = { "review": review }
    if review.verdict == "reject":
        result["revision_cycles"] = revision_cycles + 1

    return result

In [ ]:
def decide_if_should_rework_draft(state: ContentCreationState):
    review = state.get("review")
    return END if review is not None and review.verdict == "approve" else "writer"

In [ ]:
content_creation_graph_builder = StateGraph(ContentCreationState)
content_creation_graph_builder.add_node("trend_analyst", analyze_trends)
content_creation_graph_builder.add_node("writer", copywrite)
content_creation_graph_builder.add_node("editor", review)

content_creation_graph_builder.add_edge(START, "trend_analyst")
content_creation_graph_builder.add_edge("trend_analyst", "writer")
content_creation_graph_builder.add_edge("writer", "editor")
content_creation_graph_builder.add_conditional_edges("editor", decide_if_should_rework_draft, ["writer", END])

content_creation_graph = content_creation_graph_builder.compile()

In [ ]:
display_graph(content_creation_graph, Path("/content/content_creation_graph.png"))

In [ ]:
def merge_dicts(a: Dict, b: Dict):
    # IDEA: We can clear the dictionary by using "specialized" actions.
    # if len(b) == 1 and "__action__" in b:
    #     if b["__action__"] == "clear":
    #         return {}
    #     else:
    #         raise RuntimeError("Cannot merge: Unrecognized action.")
    return { **a, **b }

class MarketingContent(TypedDict):
    trend_analysis: str
    draft: str
    review: EditorReview

class MarketingCampaignState(TypedDict):
    product_brief: str

    target_platforms: List[str]
    pending_work: List[str]

    # { platform -> result }
    posts: Annotated[Dict[str, MarketingContent], merge_dicts]

In [ ]:
def orchestrate(state: MarketingCampaignState):
    target_platforms = state.get("target_platforms", [])
    posts = state.get("posts", {})

    pending_work = [platform for platform in target_platforms if platform not in posts]
    return { "pending_work": pending_work }

In [ ]:
def delegate_tasks(state: MarketingCampaignState):
    pending_work = state.get("pending_work", [])
    if not pending_work:
        return END

    return [Send("content_creation_team", { "product_brief": state.get("product_brief"), "target_platform": target_platform }) for target_platform in pending_work]

def process_content_creation_result(state: ContentCreationState):
    platform = state.get("target_platform")
    if not platform:
        raise RuntimeError("Cannot process content creation result: Missing target platform")

    trend_analysis = state.get("trend_analysis")
    if not trend_analysis:
        raise RuntimeError("Cannot process content creation result: Missing trend analysis")

    draft = state.get("draft")
    if not draft:
        raise RuntimeError("Cannot process content creation result: Missing draft")

    return {
        "posts": {
            platform: {
                # IDEA: Include review history
                "trend_analysis": trend_analysis,
                "draft": draft
            }
        }
    }

In [ ]:
in_memory_checkpointer = InMemorySaver()

In [ ]:
marketing_campaign_graph_builder = StateGraph(MarketingCampaignState)
marketing_campaign_graph_builder.add_node("orchestrator", orchestrate)
marketing_campaign_graph_builder.add_node("content_creation_team", content_creation_graph | process_content_creation_result)

marketing_campaign_graph_builder.add_edge(START, "orchestrator")
marketing_campaign_graph_builder.add_conditional_edges("orchestrator", delegate_tasks, ["content_creation_team", END])
marketing_campaign_graph_builder.add_edge("content_creation_team", "orchestrator")
marketing_campaign_graph = marketing_campaign_graph_builder.compile(checkpointer=in_memory_checkpointer)

In [ ]:
display_graph(marketing_campaign_graph, Path("/content/marketing_campaign_graph.png"))

In [ ]:
thread1_config = { "configurable": { "thread_id": "thread_1" } }
thread1_state = marketing_campaign_graph.invoke(
    input={ "product_brief": "We are launching a new eco-friendly smart water bottle. Target: gym-goers.", "target_platforms": ["Facebook", "LinkedIn"] },
    config=thread1_config
)

In [ ]:
# NOTE: We expect that there will be at least one interrupt.
print_interrupts(thread1_state["__interrupt__"])
thread1_resume = { i.id: { "decision": "approve" } for i in thread1_state["__interrupt__"]}

In [ ]:
thread1_state = marketing_campaign_graph.invoke(
    input=Command(resume=thread1_resume),
    config=thread1_config
)

In [ ]:
thread1_state